In [3]:
import h5netcdf

fn = "./13901616_nc/wT_bar_12h_blocks_true.nc"

with h5netcdf.File(fn, "r") as nc:
    print(nc)
    tlen = nc.variables["time"].shape[0]
    vlen = nc.variables["wT_bar_kC"].shape[0]
    print("time length:", nc.variables["time"][-1])
    print("data length:", vlen)
    print("equal?", tlen == vlen)

<h5netcdf.File 'wT_bar_12h_blocks_true.nc' (mode r)>
Dimensions:
    k: <h5netcdf.Dimension 'k': size 49>
    j: <h5netcdf.Dimension 'j': size 120>
    i: <h5netcdf.Dimension 'i': size 60>
    time: <h5netcdf.Dimension 'time': size 2308 (unlimited)>
Groups:
Variables:
    wT_bar_kC: ('time', 'k', 'j', 'i') float32
    time: ('time',) int64
Attributes:
    origin_iter: 22719600
    hourly_step: 144
    block_size: 12
    time_is_block_start: 1
    box_j: 10
    box_i: 10
    variable: 'wT_bar_kC'
    to_wm2: 1
    total_flux_definition: 'mean(W,12h) * interp_Zleft(mean(T,12h))'
time length: 26706096
data length: 2308
equal? True


In [2]:
import os
import numpy as np
import xarray as xr
import h5netcdf

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import read_mitgcm
import make_grid
import build_coarse_grid
from coarsen_UVWTS import coarse_field


# --------------------- your helper (kept for reference) ----------------------------------
def heat_flux(
    ds_fine: xr.Dataset,
    W: xr.Dataset,
    T: xr.Dataset,
    boundary_Z: str = "fill",
):
    """
    Compute WT (hourly): WT(t) = W(t) * interp_Zleft(T(t))
    """
    ds = xr.merge([ds_fine, W, T])
    grid_fine = make_grid.make_grid_periodic_x(ds)

    T_w = grid_fine.interp(ds["T"], "Z", to="left", boundary=boundary_Z)  # (k_l,j,i)
    WT = ds["W"] * T_w
    return WT


# --------------------- submesoscale heat flux ----------------------------------
def submeso_vertical_heat_flux(
    ds_fine: xr.Dataset,
    ds_coarse: xr.Dataset,
    ds_ocean: xr.Dataset,
    ds_ocean_c: xr.Dataset,
    WT: xr.DataArray,
    boundary_Z: str = "fill",
    box_j: int = 10,
    box_i: int = 10,
    to_wm2=True,
):
    ds = xr.merge([ds_fine, ds_ocean])
    grid_fine = make_grid.make_grid_periodic_x(ds)
    grid_coarse = make_grid.make_grid_periodic_x(ds_coarse)

    upper = ds["hFacC"]
    lower = ds["hFacC"].shift(k=-1, fill_value=0.0)
    open_frac = xr.ufuncs.minimum(upper, lower).rename({"k": "k_l"})
    Aface = (open_frac * ds["RAC"]).astype("float32")
    wet_w = Aface > 0

    # kept here as in your original function
    T_w = grid_fine.interp(ds["T"], "Z", to="left", boundary=boundary_Z)

    def _area_weighted_boxmean(A: xr.DataArray, WGT: xr.DataArray, box_j=10, box_i=10) -> xr.DataArray:
        num = (A * WGT).coarsen(j=box_j, i=box_i, boundary="trim").sum()
        den = (WGT).coarsen(j=box_j, i=box_i, boundary="trim").sum()
        return xr.where(den > 0, num / den, np.nan)

    def as_ordinal(da: xr.DataArray) -> xr.DataArray:
        da = da.reset_coords(drop=True)
        new_coords = {d: xr.DataArray(np.arange(da.sizes[d]), dims=d) for d in da.dims}
        return da.assign_coords(**new_coords)

    wT_bar_kL = _area_weighted_boxmean(WT.where(wet_w), Aface)
    w_bar_kL = ds_ocean_c["W"]
    T_bar_kC = ds_ocean_c["T"]
    T_bar_kL = grid_coarse.interp(T_bar_kC, "Z", to="left", boundary=boundary_Z)
    wT_bar_kL = as_ordinal(wT_bar_kL)

    # meso
    wT_meso_kL = (w_bar_kL * T_bar_kL)
    wT_sub_kL = (wT_bar_kL - wT_meso_kL).rename("wT_sub_kL")
    wT_bar_kL = wT_bar_kL.rename("wT_bar_kL")
    wT_meso_kL = wT_meso_kL.rename("wT_meso_kL")
    w_bar_kL = w_bar_kL.rename("w_bar_kL")
    T_bar_kL = T_bar_kL.rename("T_bar_kL")

    # switch to center
    wT_sub_kC = grid_coarse.interp(wT_sub_kL, "Z", to="center", boundary=boundary_Z).rename("wT_sub_kC")
    wT_meso_kC = grid_coarse.interp(wT_meso_kL, "Z", to="center", boundary=boundary_Z).rename("wT_meso_kC")
    wT_bar_kC = grid_coarse.interp(wT_bar_kL, "Z", to="center", boundary=boundary_Z).rename("wT_bar_kC")

    if {"j", "i"}.issubset(ds_coarse.coords):
        for da in (wT_sub_kL, wT_bar_kL, w_bar_kL, T_bar_kL, wT_sub_kC):
            if "j" in da.dims:
                da = da.assign_coords(j=ds_coarse["j"])
            if "i" in da.dims:
                da = da.assign_coords(i=ds_coarse["i"])
        wT_sub_kL, wT_bar_kL, w_bar_kL, T_bar_kL, wT_sub_kC = [
            x for x in (wT_sub_kL, wT_bar_kL, w_bar_kL, T_bar_kL, wT_sub_kC)
        ]

    if to_wm2:
        rho0, cp = 1035.0, 3994.0
        wT_meso_kC = (rho0 * cp * wT_meso_kC).assign_attrs(units="W m^-2")
        wT_bar_kC = (rho0 * cp * wT_bar_kC).assign_attrs(units="W m^-2")
        wT_sub_kC = (rho0 * cp * wT_sub_kC).assign_attrs(units="W m^-2")
    else:
        for da in (wT_sub_kL, wT_bar_kL, wT_sub_kC):
            da.attrs["units"] = "K m s^-1"

    wT_meso_kC.attrs["long_name"] = "mesoscale vertical heat flux at centers (interp from interfaces)"
    wT_bar_kC.attrs["long_name"] = "total coarse vertical heat flux at centers (interp from interfaces)"
    wT_sub_kC.attrs["long_name"] = "submesoscale vertical heat flux at centers (interp from interfaces)"

    return wT_sub_kC, wT_meso_kC, wT_bar_kC


# --------------------- loader ----------------------------------
def make_mitgcm_loader_onevar(grid_dir, read_mitgcm, var: str, dims):
    def load_snapshot(it: int) -> xr.DataArray:
        da = read_mitgcm._ocean_var(grid_dir, var, it, dims)
        da = da.drop_vars("time", errors="ignore")
        da.name = var
        return da

    return load_snapshot


# --------------------- main worker: compute + save 2 vars to 2 files (meso + bar) -----
def update_12h_submeso_flux_to_netcdf(
    start_time_index: int,
    end_time_index: int,
    load_W,  # -> DataArray (k_l,j,i)
    load_T,  # -> DataArray (k,j,i)
    out_nc_meso: str,
    out_nc_bar: str,
    ds_fine: xr.Dataset,
    ds_coarse: xr.Dataset,
    box_j: int = 10,
    box_i: int = 10,
    block_size: int = 12,
    step: int = 144,
    time_name: str = "time",
    boundary_Z: str = "fill",
    to_wm2: bool = True,
    dtype=np.float32,
    skip_missing: bool = False,
    print_every: int = 20,
):
    start_time_index = int(start_time_index)
    end_time_index = int(end_time_index)
    step = int(step)

    if end_time_index < start_time_index:
        raise ValueError("end_time_index must be >= start_time_index")

    os.makedirs(os.path.dirname(out_nc_meso) or ".", exist_ok=True)
    os.makedirs(os.path.dirname(out_nc_bar) or ".", exist_ok=True)

    block_span = block_size * step
    origin_iter = start_time_index

    def _read_existing_meta(path: str):
        nonlocal origin_iter
        if not os.path.exists(path):
            return set()
        with h5netcdf.File(path, "r") as nc:
            saved_times = set()
            if time_name in nc.variables:
                saved_times = set(map(int, nc.variables[time_name][:].tolist()))
            if "origin_iter" in nc.attrs:
                origin_iter = int(nc.attrs["origin_iter"])
            if "box_j" in nc.attrs and int(nc.attrs["box_j"]) != int(box_j):
                raise ValueError("box_j mismatch with existing file")
            if "box_i" in nc.attrs and int(nc.attrs["box_i"]) != int(box_i):
                raise ValueError("box_i mismatch with existing file")
            return saved_times

    saved_meso = _read_existing_meta(out_nc_meso)
    saved_bar = _read_existing_meta(out_nc_bar)
    saved_all = saved_meso & saved_bar

    n0 = int(np.ceil((start_time_index - origin_iter) / block_span)) if start_time_index > origin_iter else 0
    n1 = int(np.floor((end_time_index - origin_iter - (block_size - 1) * step) / block_span))
    if n1 < n0:
        print("Not enough hourly data to form even one full 12-hour block yet.")
        return

    grid_fine = None

    def _init_file_onevar(out_path: str, varname: str, example_flux_kC: xr.DataArray):
        spatial_dims = tuple(example_flux_kC.dims)
        spatial_sizes = {d: int(example_flux_kC.sizes[d]) for d in spatial_dims}

        with h5netcdf.File(out_path, "w") as nc:
            nc.dimensions[time_name] = None
            for d, n in spatial_sizes.items():
                nc.dimensions[d] = n

            nc.attrs["origin_iter"] = int(origin_iter)
            nc.attrs["hourly_step"] = int(step)
            nc.attrs["block_size"] = int(block_size)
            nc.attrs["time_is_block_start"] = 1
            nc.attrs["box_j"] = int(box_j)
            nc.attrs["box_i"] = int(box_i)
            nc.attrs["variable"] = varname
            nc.attrs["to_wm2"] = int(bool(to_wm2))
            nc.attrs["total_flux_definition"] = "mean(W,12h) * interp_Zleft(mean(T,12h))"

            tvar = nc.create_variable(time_name, (time_name,), dtype="i8", chunks=(1,), maxshape=(None,))
            tvar.attrs["long_name"] = "start index of each 12-hour averaging block"
            tvar.attrs["units"] = "model_iteration_index"

            dims = (time_name,) + spatial_dims
            chunks = (1,) + tuple(spatial_sizes[d] for d in spatial_dims)
            maxshape = (None,) + tuple(spatial_sizes[d] for d in spatial_dims)

            v = nc.create_variable(varname, dims, dtype="f4", chunks=chunks, maxshape=maxshape)
            v.attrs["long_name"] = example_flux_kC.attrs.get("long_name", varname)
            v.attrs["units"] = example_flux_kC.attrs.get("units", "W m^-2" if to_wm2 else "K m s^-1")

    def _append_onevar(nc: h5netcdf.File, varname: str, tvalue: int, data: xr.DataArray):
        tvar = nc.variables[time_name]
        write_idx = int(tvar.shape[0])

        v = nc.variables[varname]
        spatial_shape = v.shape[1:]
        v._h5ds.resize((write_idx + 1,) + spatial_shape)
        v[write_idx, ...] = data.values.astype(np.float32, copy=False)

        tvar._h5ds.resize((write_idx + 1,))
        tvar[write_idx] = int(tvalue)

    appended = skipped_saved = skipped_missing = 0
    nc_meso = None
    nc_bar = None

    try:
        for n in range(n0, n1 + 1):
            s = origin_iter + n * block_span

            if print_every and ((n - n0) % print_every == 0):
                print(f"[try] n={n}/{n1} block_start={s}", flush=True)

            if s in saved_all:
                skipped_saved += 1
                continue

            block_iters = [s + m * step for m in range(block_size)]

            W0 = T0 = None
            sumW = sumT = None

            try:
                for it in block_iters:
                    W = load_W(int(it)).astype(dtype)
                    T = load_T(int(it)).astype(dtype)

                    if W0 is None:
                        W0, T0 = W, T
                        sumW = W.values.astype(dtype, copy=True)
                        sumT = T.values.astype(dtype, copy=True)

                        if grid_fine is None:
                            ds_tmp = xr.merge([ds_fine, xr.Dataset({"W": W0, "T": T0})])
                            grid_fine = make_grid.make_grid_periodic_x(ds_tmp)
                    else:
                        sumW += W.values
                        sumT += T.values

            except Exception as e:
                if skip_missing:
                    skipped_missing += 1
                    print(f"  -> missing/unreadable snapshot in block_start={s}, skip ({e})", flush=True)
                    continue
                raise

            # 12h means of W and T
            W_12mean = W0.copy(data=(sumW / block_size)).astype("float32")
            T_12mean = T0.copy(data=(sumT / block_size)).astype("float32")

            # changed part: total flux from 12h-mean fields
            T_12mean_w = grid_fine.interp(T_12mean, "Z", to="left", boundary=boundary_Z)
            WT_12mean = (W_12mean * T_12mean_w).astype("float32")
            WT_12mean.name = "WT"

            # build ds_ocean and ds_ocean_c
            ds_ocean = xr.Dataset(dict(W=W_12mean, T=T_12mean))
            W_coarse = coarse_field("W", W_12mean, ds_fine, box_j=box_j, box_i=box_i)
            T_coarse = coarse_field("T", T_12mean, ds_fine, box_j=box_j, box_i=box_i)
            ds_ocean_c = xr.Dataset(dict(W=W_coarse, T=T_coarse))

            # compute fluxes for this block
            wT_sub_kC, wT_meso_kC, wT_bar_kC = submeso_vertical_heat_flux(
                ds_fine=ds_fine,
                ds_coarse=ds_coarse,
                ds_ocean=ds_ocean,
                ds_ocean_c=ds_ocean_c,
                WT=WT_12mean,
                boundary_Z=boundary_Z,
                box_j=box_j,
                box_i=box_i,
                to_wm2=to_wm2,
            )

            wT_meso_kC = wT_meso_kC.astype("float32")
            wT_bar_kC = wT_bar_kC.astype("float32")

            if not os.path.exists(out_nc_meso):
                _init_file_onevar(out_nc_meso, "wT_meso_kC", wT_meso_kC)
            if not os.path.exists(out_nc_bar):
                _init_file_onevar(out_nc_bar, "wT_bar_kC", wT_bar_kC)

            if nc_meso is None:
                nc_meso = h5netcdf.File(out_nc_meso, "a")
            if nc_bar is None:
                nc_bar = h5netcdf.File(out_nc_bar, "a")

            _append_onevar(nc_meso, "wT_meso_kC", s, wT_meso_kC)
            _append_onevar(nc_bar, "wT_bar_kC", s, wT_bar_kC)

            saved_meso.add(s)
            saved_bar.add(s)
            saved_all = saved_meso & saved_bar

            appended += 1

    finally:
        for fh in (nc_meso, nc_bar):
            if fh is not None:
                fh.close()

    print(
        f"\nUpdated NetCDFs (saved meso + bar):\n"
        f"  meso: {out_nc_meso}\n"
        f"  bar : {out_nc_bar}\n"
        f"  appended: {appended}\n"
        f"  skipped (already saved): {skipped_saved}\n"
        f"  skipped (missing hours): {skipped_missing}\n"
        f"  origin_iter={origin_iter}, step={step}, block_size={block_size}\n"
        f"  coarse box: box_j={box_j}, box_i={box_i}",
        flush=True,
    )


# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":
    path_scratch = "/scratch/jczhang/"
    path_MITgcm = "MITgcm_baroclinic_instability_old/MITgcm/verification/tutorial_reentrant_channel/run"
    grid_dir = path_scratch + path_MITgcm

    ds_fine = read_mitgcm.read_mitgrid_staggered(grid_dir)

    box_j, box_i = 10, 10
    ds_coarse = build_coarse_grid.build_coarse_grid_staggered(ds_fine, box_j=box_j, box_i=box_i)

    dims_map = {
        "W": ("k_l", "j", "i"),
        "T": ("k", "j", "i"),
    }

    start_time_index, end_time_index, step = 22719600, 26707984, 144

    load_W = make_mitgcm_loader_onevar(grid_dir, read_mitgcm, var="W", dims=dims_map["W"])
    load_T = make_mitgcm_loader_onevar(grid_dir, read_mitgcm, var="T", dims=dims_map["T"])

    out_nc_meso = "./13901616_nc/wT_meso_12h_blocks_true.nc"
    out_nc_bar = "./13901616_nc/wT_bar_12h_blocks_true.nc"

    update_12h_submeso_flux_to_netcdf(
        start_time_index=start_time_index,
        end_time_index=end_time_index,
        load_W=load_W,
        load_T=load_T,
        out_nc_meso=out_nc_meso,
        out_nc_bar=out_nc_bar,
        ds_fine=ds_fine,
        ds_coarse=ds_coarse,
        box_j=box_j,
        box_i=box_i,
        block_size=12,
        step=step,
        time_name="time",
        skip_missing=False,
        print_every=1,
    )

[try] n=0/2307 block_start=22719600
[try] n=1/2307 block_start=22721328
[try] n=2/2307 block_start=22723056
[try] n=3/2307 block_start=22724784
[try] n=4/2307 block_start=22726512
[try] n=5/2307 block_start=22728240
[try] n=6/2307 block_start=22729968
[try] n=7/2307 block_start=22731696
[try] n=8/2307 block_start=22733424
[try] n=9/2307 block_start=22735152
[try] n=10/2307 block_start=22736880
[try] n=11/2307 block_start=22738608
[try] n=12/2307 block_start=22740336
[try] n=13/2307 block_start=22742064
[try] n=14/2307 block_start=22743792
[try] n=15/2307 block_start=22745520
[try] n=16/2307 block_start=22747248
[try] n=17/2307 block_start=22748976
[try] n=18/2307 block_start=22750704
[try] n=19/2307 block_start=22752432
[try] n=20/2307 block_start=22754160
[try] n=21/2307 block_start=22755888
[try] n=22/2307 block_start=22757616
[try] n=23/2307 block_start=22759344
[try] n=24/2307 block_start=22761072
[try] n=25/2307 block_start=22762800
[try] n=26/2307 block_start=22764528
[try] n=27/